# Assignment - Song Classification

Welcome to your last Assignment! You will build your own song classifer using k-nearest neighbors.

You will build a classifier that guesses whether a song is hip-hop or country, using only the numbers of times words appear in the song's lyrics.  By the end of the project, you should know how to:

1. Clean and organize a dataset used to test a machine learning model
2. Build a k-nearest neighbors classifier
3. Test a classifier on data

**Advice.** Develop your answers incrementally. To perform a complicated table manipulation, break it up into steps, perform each step on a different line, give a new name to each result, and check that each intermediate result is what you expect. You can add any additional names or functions you want to the provided cells. 

To get started, load `datascience`, `numpy`, `plots`, and `gofer`.

In [1]:
# Run this cell to set up the notebook, but please don't change it.
import numpy as np
import math
import datascience
from datascience import *

# These lines set up the plotting functionality and formatting.
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import warnings
warnings.simplefilter("ignore")

# 1. The Data set

The dataset is a table of songs, each with a name, artist, and genre. We will be trying to predict each song's genre.

Only attribute we will use to predict genre of a song are its lyrics. List of just under 5000 words that might occur in a song. For each song, our dataset tells us the frequency w/ each of these words occurs in that song. All words have been converted to lowercase. 

Run the cell below to read the `lyrics` table. **It may take up to a minute to load.**

In [ ]:
lyrics = Table.read_table('lyrics.csv')
lyrics.show(3)


The output above prints a rows of the dataset. Along with the Title, Artist and Genre we have the proportion of times cetrain words appear in the song. For example if the song has 100 words and 'like' appreas 2 times, the proportion in the table will display as $\frac{2}{100} = 0.02$. If a word does not appear in the song, the proportion is recorded as $0$. 

Our dataset doesn't contain all information about a song.  For example, it doesn't describe the order of words in the song, let alone the melody, instruments, or rhythm. Nonetheless, you may find that word frequencies alone are sufficient to build an accurate genre classifier.

There is a song called "In Your Eyes" in the dataset. Use python to identify the artist of the song and the proportion of times the word 'love' appears in the song.

The word "like" appears twice:  $\frac{2}{168} \approx 0.0119$ of the words in the song. The word "love" appears 10 times: $\frac{10}{168} \approx 0.0595$ of the words. The word "the" doesn't appear at all.

All titles are unique. The row_for_title function provides fast access to one row for each title

In [ ]:
title_index = lyrics.index_by('Title')
def row_for_title(title):
    """Return the row for a title, similar to the following expression (but faster) 
        lyrics.where('Title',title).row(0)
    """
    return title_index.get(title)[0]

for example, the fastest way to find the frequency of "love" in the song In your eyes is to access the 'love' item from its row:

In [ ]:
row_for_title('In Your Eyes').item('love')

### Question 1.1:
Set expected_row_sum to the number that you will result from summing all proportions in each row, excluding the first 3 columns

In [ ]:
expected_row_sum = ...

The following cell below generates a histogram of actual row sums. It should confirm your answer above, perhaps with a small amount of error:

In [ ]:
Table().with_column('sums', lyrics.drop([0, 1, 2]).apply(sum)).hist(0)

The dataset was extracted from the Million Song Dataset. Specifically, datasets from musiXmatch and Last.fum

The counts of common words in the lyrics for all of these songs are provided by the musiXmatch dataset (called a bag-of-words format). We converted the words to lowercase, removed the explicit words, and converted the counts to frequencies

The Last.fm dataset contains multiple tags for each song in the Million Song Dataset. Some tags are genre-related such as 'pop', 'punk', 'rock', etc. To construct the Genre column, first we extracted songs with Last.fm tags that included the words 'country', or both 'hip' and 'hop'. These songs were then cross-referenced witht he musiXmatch dataset, and only songs with musiXmatch lyrics were placed in the dataset. 
   

In [ ]:
print('Words with frequencies:', lyrics.drop('Title','Artist','Genre').num_columns)
print('Words with genres:', lyrics.num_rows)

# 1.1: Word Stemming
The columsn other than Title, Artist, Genre in the lyrics table are all words in some of the songs in our dataset. Some of those names have been *Stemmed* or abbreviated heuristically, in an ettempt to make different *inflected* forms of the same base word into the same string.

For example, the column "manag" is the sum of proportions of the words 'manage', 'manager', etc. 

Stemming makes it a little tricky to search for words you want to use, so we have provided another table that will let us see examples of unstemmed versions of each stemmed word. Run the following code below to load the table

In [ ]:
vocab_mapping = Table.read_table('mxm_reverse_mapping_safe.csv')
stemmed = np.take(lyrics.labels, np.arange(3, len(lyrics.labels)))
vocab_table = Table().with_column('Stem', stemmed).join('Stem', vocab_mapping)
vocab_table.take(np.arange(1100, 1110))

### Question 1.1.1:
Assign unchanged to the percentage of words in the vocab_table that are teh same as their stemmed form (such as devour above)

*Hint*: Try using where and comparing the number of rows in a table of only unchanged vocabulary with the number of rows in vocab_table

In [ ]:
...

### Question 1.1.2:
Assign stememd_message to the stemmed version of the word 'message'.

In [ ]:
stemmed_message = ...

### Question 1.1.3:
Assign unstemmed_singl to the word in vocab_table that has 'singl' as its stemmed form.

In [ ]:
unstemmed_singl = ...
unstemmed_singl

# 1.2: Splitting the Dataset
We're going to use our lyrics dataset for two purposes:
1) First, we want to *train* song genre classifiers.
2) Second, we want to *test* the performance of our classifiers

Hence, we need two different datasets: *training* and *test*

The purpose of a classifier is to classify unseen data that is similar to the training data. Therefore, we must ensure that there are no songs that appear in both sets.

We do so by splitting the dataset randomly. The dataset has already been permuted randomly, so its easy to split.

We just take the top for training and the rest for test.

Run the following code below (w/o changing it) to sperate the dataset into two tables:

In [ ]:

# Here we have defined the proportion of our data
# that we want to designate for training as 11/16ths
# of our total dataset.  5/16ths of the data is
# reserved for testing.

training_proportion = 11/16

num_songs = lyrics.num_rows
num_train = int(num_songs * training_proportion)
num_valid = num_songs - num_train

train_lyrics = lyrics.take(np.arange(num_train))
test_lyrics = lyrics.take(np.arange(num_train, num_songs))

print("Training: ",   train_lyrics.num_rows, ";",
      "Test: ",       test_lyrics.num_rows)

### Question 1.2.1:
Draw a horizontal bar chart with two bars that show the proportion of Country songs in each dataset. Complete the function country_proportion first, it should help you create the bar chart.

In [ ]:
def country_proportion(table):
    """Return the proportion of songs in a table that have a Country genre"""
    ...
    
Prop_country = Table().with_columns()


# 2. K-Nearest Neighbors - a Guided Example

K-Nearest Neighbors (k-NN) is a classification algorithm

Given some *attributes* (also called *features*) of an unseen example, it decideds wehther that exmaple belongs to one or the other of two categories based on its similarity to previously seen examples. 

Predicting the category of an example is called *labeling* and th predicted caregory is also called a *label*

An attribute (feature) we have about each song is the *proportion of times a particular word appears in the lyrics*, and the labels are two music genres: Country and Hiphop.

The algorith requires many previously seen examples for which both the attributes and labels are known: thats the train_lyrics table

To build understanding, we're going to visualize the algorith instead of just describing it

# 2.1: Classifying a song

in k-NN, we classify a song by: finding the k songs in the *training set* that are most similar according to the features we choose. We call those songs w/ similar features the *nearest neighbors*. The k-NN algoorith assigns the most common category among its k-nearest neighbors

Let's limit ourselves to just 2 features for now, so we can plot each song. The features we will use are the proportions of the words "like" and "love" in the lyrics. Taking the song "In Your Eyes" (in the test set), 0.0119 of its words are "like" and 0.0595 are "love". This song appears in the test set, so let's imagine that we don't yet know its genre.

First, we need to make our notion of similarity more precise. We will say that the distance between two songs is the straight-line distance between them when we plot their features in a scatter diagram. This distance is called the Euclidean ("yoo-KLID-ee-un") distance.

For example, in the song Insane in the Brain (in the training set), 0.0203 of all the words in the song are "like" and 0 are "love". Its distance from In Your Eyes on this 2-word feature set is $\sqrt{(0.0119 - 0.0203)^2 + (0.0595 - 0)^2} \approx 0.06$. (If we included more or different features, the distance could be different.)

A third song, Sangria Wine (in the training set), is 0.0044 "like" and 0.0925 "love".

The function below creates a plot to display the "like" and "love" features of a test song and some training songs. As you can see in the result, In Your Eyes is more similar to Sangria Wine than to Insane in the Brain.

In [ ]:
# Just run this cell.

def plot_with_two_features(test_song, training_songs, x_feature, y_feature):
    """Plot a test song and training songs using two features."""
    test_row = row_for_title(test_song)
    distances = Table().with_columns(
            x_feature, [test_row.item(x_feature)],
            y_feature, [test_row.item(y_feature)],
            'Color',   ['Unknown'],
            'Title',   [test_song]
        )
    for song in training_songs:
        row = row_for_title(song)
        distances.append([row.item(x_feature), row.item(y_feature), row.item('Genre'), song])
    distances.scatter(x_feature, y_feature, group='Color', labels='Title', s=200)
    
training = ["Sangria Wine", "Insane In The Brain"]
plot_with_two_features("In Your Eyes", training, "like", "love")

### Question 2.1.1: 
Compute the distance between the two country songs, In Your Eyes and Sangria Wine, using the like and love features only. Assign it the name country_distance.

Note: If you have a row object, you can use item to get a value from a column by its name. For example, if r is a row, then r.item("Genre") is the value in column "Genre" in row r.

Note 2: You can quickly get the row from the lyrics table via row_for_title. For example, if "Insane In The Brain" is the song title, then row_for_title("Insane In The Brain") is the row object for this song.

In [ ]:
in_your_eyes = row_for_title("In Your Eyes")
sangria_wine = row_for_title("Sangria Wine")
...

country_distance = ...
country_distance

### Question 2.1.2:
Complete the function distance_two_features that computes the Euclidean distance between any two songs, using two features

the last two lines call your function to show that *Lookin' for Love* is closer to *In Your Eyes* than *Insane in the Brain*

In [ ]:
def distance_two_features(title0,title1,x_feature,y_feature):
    """Compute the distance between two songs with titles title0 and title1
    
    Only the features named x_feature and y_feature are used when computing the distance.
    """
    row0 = row_for_title(title0)
    row1 = ...
    
    distance0 = make_array(row0.item(x_feature), row0.item(y_feature))
    distance1 = ...
    return 

for song in make_array("Lookin' for Love", "Insane In The Brain"):
    song_distance = distance_two_features(song, "In Your Eyes", "like", "love")
    print(song, 'distance:\t', song_distance)

### Question 2.1.3:
Define the function distane_from_in_your_eyes so that it works as describes in its documentation:

In [ ]:
def distance_from_in_your_eyes(title):
    """The distance between the given song and "In Your Eyes", based on the features "like" and "love".
    
    This function takes a single argument:
      title: A string, the name of a song.
    """
    ...

### Question 2.1.4:
Using the features 'like' and 'love', what are the names and genres of the 7 songs in the *training set* closest to 'In Your Eyes'?

To answer this question, make a table named close_songs with those 7 songs w/ columns "Title", "Genre", "like", "love", as well as "distance" from 'In Your Eyes'.

Table should be sorted in ascending order by distance

In [ ]:

close_songs = ...
close_songs

### Question 2.1.5:
Define the function most_common so that it words as described in the documentation below:


In [ ]:

def most_common(label, table):
    """The most common element in a column of a table.
    
    This function takes two arguments:
      label: The label of a column, a string.
      table: A table.
     
    It returns the most common value in that column of that table.
    In case of a tie, it returns any one of the most common values
    """
    most = table.group(label).sort(1, descending=True).column(0).item(0)
    return most
    

# Calling most_common on your table of 7 nearest neighbors classifies
# "In Your Eyes" as a country song, 4 votes to 3.
most_common('Genre', close_songs)

Congratulations are in order -- you've classified your first song!

# Great job on working through this assignment. You are all done!